# Serve the E4B adapter on Colab T4 — web UI (HF nf4, full fidelity)

Serves your trained E4B LoRA at the SAME nf4 4-bit precision it was trained on (no GGUF
quant degradation), on a free Colab **T4 (16GB)** which fits E4B 4-bit with room — then
exposes the browser chat UI over an ngrok tunnel. This is the truest read of model quality
(q4_0 GGUF would confound it; we've seen q4_0 fabricate numbers).

> Serve-only: the eval/probe cells (train-row reproduction, plan probe, routing confusion
> matrix) were stripped so this gets to a live URL fast. They live in the Kaggle benchmark
> notebook (`kaggle_benchmark.ipynb`) — run them there.

## Run order
**1** config → **2** GPU check → **3** clone/pull → **4** deps → **5** base download →
**6** adapter download (6b = upload alt) → **7** SERVE (prints the browser URL).

The serve cell loads the model in a **subprocess** service (once, ~1-2 min), then runs the
weightless web app against it — so the T4 holds exactly **one** E4B. Re-running the serve
cell only restarts the cheap app; the model service is reused.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Add Colab Secrets:
`HF_TOKEN` (HF read) and `NGROK_TOKEN` (free from ngrok.com → authtoken).

**A/B the harness:** edit the two toggles in Cell 1 (CHESS_THIN_HARNESS, CHESS_BOARD_HOOK), then re-run the SERVE cell. Only the weightless app restarts (no model reload), so each flip applies in seconds. The live config is printed at the bottom of the serve cell.

In [ ]:
# Cell 1 - config
import os
BASE_REPO = "unsloth/gemma-4-E4B-it"   # the base your adapter was trained on
REPO_URL  = "https://github.com/RyanDev1st/llm-and-engine.git"
BRANCH    = "master"   # production: has the session sidebar + neural opponent (merged from feat/chat-sessions)

if os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    WORKDIR = "A:/Download/llm_tool_calling_research_workspace"
    ADAPTER_DIR = "A:/Download/gemma4_chess_e4b_kaggle (1)"
else:
    WORKDIR = "/content/llm-and-engine"
    ADAPTER_DIR = "/content/adapter"

BASE_DIR  = f"{WORKDIR}/src/llm/models/gemma4_e4b"
print("base:", BASE_REPO, "| adapter:", ADAPTER_DIR)

# --- harness A/B toggles (flip a value, then RE-RUN the serve cell to apply; no model reload) ---
# Defaults reproduce the CURRENT live behavior. The 2x2 to try is in
# docs/findings/2026-06-24-harness-live-vs-benchmark-gap.md.
THIN_HARNESS = "0"   # "1" = trust the model: drop coverage force-routing + the per-turn self-verify probe + ask-back re-gen
BOARD_HOOK   = "1"   # "0" = train parity: stop injecting the LIVE BOARD line into the system prompt
print("toggles -> CHESS_THIN_HARNESS=", THIN_HARNESS, " CHESS_BOARD_HOOK=", BOARD_HOOK)

# --- model BACKEND (re-run the EXPORT cell + the SERVE cell after changing) ---
# "hf"   = transformers nf4 base + adapter (current; supports the base-vs-trained dual compare)
# "gguf" = llama.cpp Q5_K_M/Q6_K (faster decode; single merged file, no live base compare)
BACKEND    = "hf"        # "hf" | "gguf"
GGUF_QUANT = "Q5_K_M"    # "Q5_K_M" (balanced) | "Q6_K" (near-lossless) — only used when BACKEND="gguf"
GGUF_PATH  = ""          # filled by the export cell
print("BACKEND =", BACKEND, "| GGUF_QUANT =", GGUF_QUANT)


In [ ]:
# Cell 2 - GPU check (want a T4 16GB)
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 - clone / pull repo (prints HEAD so you can confirm the latest commit landed)
import os, subprocess
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(["git","-C",WORKDIR,"fetch","origin",BRANCH], check=True, env=env)
    subprocess.run(["git","-C",WORKDIR,"reset","--hard",f"origin/{BRANCH}"], check=True, env=env)  # force latest
head = subprocess.run(["git","-C",WORKDIR,"log","-1","--oneline"], capture_output=True, text=True).stdout.strip()
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("HEAD:", head)
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps (serve path + ngrok + stockfish for full chess tools)
import subprocess, sys, shutil
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("-U","transformers","accelerate","peft","bitsandbytes","sentencepiece","protobuf","python-chess","pyngrok")
if shutil.which("apt-get"):
    subprocess.run(["apt-get","-qq","install","-y","stockfish"])  # chess tools (optional)
import transformers, peft, bitsandbytes
print("transformers",transformers.__version__,"| peft",peft.__version__,"| bnb",bitsandbytes.__version__)

In [ ]:
# Cell 5 - HF login + download the E4B base (~9GB; once)
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login(os.environ["HF_TOKEN"])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE_DIR, "->", sorted(os.listdir(BASE_DIR))[:8])

In [ ]:
# Cell 6 - download the adapter from Hugging Face (FORMAT-FIXED run = ckpt-v4)
# v4 run: all-linear + FORMAT_WEIGHT=8.0 on the control tags + the serve-side </skill>
# stop fix. best/ = lowest-val checkpoint. val is NOT the judge here — Cell 7 (train-row
# free-gen probe) and Cell 7c (routing confusion matrix + precision) are.
#   TAG='best'       -> lowest-val adapter (PROBE/SERVE THIS)
#   TAG='checkpoint' -> latest step (only for resume)
import os
from huggingface_hub import snapshot_download

ADAPTER_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v4"   # current run (v3/v2/v1 superseded)
TAG = "best"                                            # lowest-val checkpoint

if not os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    print(f"Downloading adapter from HF: {ADAPTER_REPO} :: {TAG}/")
    snapshot_download(repo_id=ADAPTER_REPO, local_dir="/content/hf_adapter_v4",
                      allow_patterns=[f"{TAG}/*"])
    ADAPTER_DIR = f"/content/hf_adapter_v4/{TAG}"

need = ("adapter_model.safetensors", "adapter_config.json")
ok = all(os.path.exists(f"{ADAPTER_DIR}/{f}") for f in need)
if os.path.exists(ADAPTER_DIR):
    print("adapter at", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR))
    # sanity: all-linear adapter MUST have MLP modules (gate/up/down), not just attn
    import json as _j
    cfg = _j.loads(open(f"{ADAPTER_DIR}/adapter_config.json").read())
    print("target_modules:", cfg.get("target_modules"), "| r:", cfg.get("r"),
          "| alpha:", cfg.get("lora_alpha"), "| dropout:", cfg.get("lora_dropout"))
else:
    print("adapter at", ADAPTER_DIR, "-> NOT FOUND")
assert ok, f"adapter not found at {ADAPTER_DIR}"

In [ ]:
# Cell 6b - (alt) upload the adapter instead of Drive. Zip best/ locally, upload here.
# from google.colab import files; import zipfile, io, os
# up = files.upload()                       # pick best.zip
# name = next(iter(up))
# ADAPTER_DIR = "/content/best"
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content")
# print(os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 6c - (optional) EXPORT a GGUF quant to serve. Runs ONLY when BACKEND="gguf".
# FAST by construction: a PREBUILT llama-cpp-python wheel (no source compile), llama-quantize built
# CPU-only (GGML_CUDA=OFF -> no slow nvcc), and ONLY the converter's deps (not llama.cpp's full
# requirements.txt, which reinstalls torch). First run ~5-8 min (the bf16 merge dominates); switching
# quant after reuses the f16 merge (~1-2 min). Each step prints, so it is never a silent hang.
if not BACKEND.lower().startswith("gguf"):
    GGUF_PATH = ""
    print("BACKEND=hf -> skipping GGUF export")
else:
    import os, sys, time, shutil, subprocess
    from pathlib import Path
    def sh(c, **kw): print("  $", c, flush=True); subprocess.run(c, shell=True, check=True, **kw)
    t0 = time.time(); os.environ["CHESS_HF_BASE"] = BASE_DIR
    print("[1/5] converter deps + PREBUILT llama-cpp-python (no source compile)...", flush=True)
    sh("pip -q install gguf sentencepiece protobuf")
    # prebuilt CUDA wheel for SERVING; if the index has no match, fall back to the CPU wheel (still no compile)
    sh("pip -q install llama-cpp-python "
       "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 "
       "|| pip -q install llama-cpp-python")
    LC = "/content/llamacpp"
    if not Path(LC + "/convert_hf_to_gguf.py").exists():
        print("[2/5] fetching llama.cpp source (for the converter + quantize)...", flush=True)
        sh(f"git clone --depth 1 https://github.com/ggerganov/llama.cpp {LC}")
    QBIN = f"{LC}/build/bin/llama-quantize"
    if not Path(QBIN).exists():
        print("[3/5] building CPU-only llama-quantize (GGML_CUDA=OFF -> ~1-2 min, no nvcc)...", flush=True)
        sh(f"cmake -S {LC} -B {LC}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_SERVER=OFF")
        sh(f"cmake --build {LC}/build --target llama-quantize -j 2")
    sys.path.insert(0, f"{WORKDIR}/src/llm")
    from llm_training.export_gguf import merge, out_names
    p = out_names(Path(WORKDIR), Path(BASE_DIR), GGUF_QUANT)
    if not p["quant"].exists():
        if not p["f16"].exists():
            print("[4/5] merge adapter -> bf16 (the slow step) + convert to f16 gguf...", flush=True)
            merge(Path(ADAPTER_DIR), p["merged"])
            subprocess.run([sys.executable, f"{LC}/convert_hf_to_gguf.py", str(p["merged"]),
                            "--outfile", str(p["f16"]), "--outtype", "f16"], check=True)
            shutil.rmtree(p["merged"], ignore_errors=True)     # free the ~16GB bf16 merge
        print(f"[5/5] quantize -> {GGUF_QUANT} ...", flush=True)
        subprocess.run([QBIN, str(p["f16"]), str(p["quant"]), GGUF_QUANT], check=True)
    GGUF_PATH = str(p["quant"])
    print(f"GGUF ready ({time.time()-t0:.0f}s):", GGUF_PATH, os.path.getsize(GGUF_PATH)//(1<<20), "MB", flush=True)


In [ ]:
# Cell 7 - SERVE the web UI (browser chat via ngrok). The web app loads the model in a SUBPROCESS
# service (loaded ONCE, ~1-2 min), then runs the weightless app against it. Re-running this cell only
# restarts the cheap app (the model service is reused). This is the only cell needed to go live.
import os, sys, subprocess, time, json, urllib.request
SVC = 'http://127.0.0.1:7861'

# Free any in-kernel model first so the subprocess service is the ONLY resident E4B (a T4 fits
# exactly one). This serve-only notebook never loads one in-kernel, so it's just a safety guard.
if globals().get('MODEL') is not None:
    del globals()['MODEL']
try:
    import gc, torch; gc.collect(); torch.cuda.empty_cache()
except Exception:
    pass

def _health():
    try: return json.load(urllib.request.urlopen(SVC+'/health', timeout=5)).get('ok')
    except Exception: return False

def _running_is_gguf():
    import urllib.request, json as _j
    try:
        h = _j.load(urllib.request.urlopen(SVC+'/health', timeout=5))
        return h.get('adapter') is False           # GGUF service has no adapter toggle
    except Exception:
        return None

def ensure_service():
    want_gguf = BACKEND.lower().startswith('gguf')
    desired_id = (GGUF_PATH or 'gguf') if want_gguf else 'hf'   # the EXACT model we want (so Q5->Q6 restarts)
    if _health():
        if _running_is_gguf() == want_gguf and globals().get('_SERVED_ID') == desired_id:
            print('model service already UP - reusing'); return
        print(f'model change -> restarting service (id={desired_id})', flush=True)
        import subprocess as _sp; _sp.run(['pkill','-f','backend.model_server']); import time as _t; _t.sleep(3)
    globals()['_SERVED_ID'] = desired_id
    env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR,
         'CHESS_BACKEND':BACKEND,'CHESS_GGUF_PATH':(GGUF_PATH or '')}
    log=open('/content/model_server.log','w')
    globals()['_MSVC']=subprocess.Popen([sys.executable,'-u','-m','backend.model_server',ADAPTER_DIR],
                                        cwd=f'{WORKDIR}/src/llm',env=env,stdout=log,stderr=subprocess.STDOUT)
    print('starting model service with', ADAPTER_DIR, '(loads ONCE, ~1-2 min)...')
    for _ in range(180):
        time.sleep(2)
        if globals()['_MSVC'].poll() is not None:
            print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service crashed')
        if _health(): print('model service UP'); return
    print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service did not come up')

ensure_service()                      # start the one model service (or reuse it)
env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR,
     'CHESS_MODEL_SERVER':SVC,'CHESS_HOST':'127.0.0.1','CHESS_PORT':'7860',
     'CHESS_THIN_HARNESS':globals().get('THIN_HARNESS','0'),'CHESS_BOARD_HOOK':globals().get('BOARD_HOOK','1')}
subprocess.run(['pkill','-f','backend.server']); time.sleep(2)   # restart app only (cheap, weightless)
globals()['_APP']=subprocess.Popen([sys.executable,'-u','-m','backend.server'],cwd=f'{WORKDIR}/src/llm',
                  env=env,stdout=open('/content/app_server.log','w'),stderr=subprocess.STDOUT)
time.sleep(5)
from pyngrok import ngrok
ngrok.kill()                          # drop old tunnels (free tier allows 1)
try:
    from google.colab import userdata; ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
except Exception:
    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
url = ngrok.connect(7860)
print('==============================================')
print('  OPEN THIS IN YOUR BROWSER TO CHAT:')
print('  ', url)
print('==============================================')
print('harness config -> CHESS_THIN_HARNESS=%s CHESS_BOARD_HOOK=%s' % (env['CHESS_THIN_HARNESS'], env['CHESS_BOARD_HOOK']))
print('--- app server log ---')
print(open('/content/app_server.log').read()[-1500:])